In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import scipy

from scipy.stats import norm

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

import statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf

### Loading in Yearly Data

In [2]:
month = "May"
year = "2019"

may19_data = load_dataset(month, year)[2]
may19_data_temporal_melted = load_dataset(month, year)[3]

x_19 = load_dataset(month, year)[0]
y_19 = load_dataset(month, year)[1]
may19_data.head()

NameError: name 'load_dataset' is not defined

#### 2021

In [ ]:
month = "May"
year = "2021"

may21_data = load_dataset(month, year)[2]
may21_data_temporal_melted = load_dataset(month, year)[3]

x_21 = load_dataset(month, year)[0]
y_21 = load_dataset(month, year)[1]
may21_data.head()

In [ ]:
data = pd.read_excel("E3 ACC Data/CZ7 Marginal Data.xlsx")
data.head()

In [ ]:
columns = [col for col in data.columns if col not in ['Year', 'Month','Day','Hour']]
columns

In [ ]:
data['Total'] = data[columns].sum(axis=1)
data.head()

In [ ]:
# # This will try conversion and flag errors
# def try_parse(row):
#     try:
#         return pd.to_datetime(f"{row['Year']}-{row['Month']}-{row['Day']} {row['Hour']}:00")
#     except:
#         return None

# # Apply to each row
# data["datetime"] = data.apply(try_parse, axis=1)

# # Find the problematic rows
# invalid_rows = data[data["datetime"].isnull()]
# print(invalid_rows)

#### Visualizing Yearly Data

In [ ]:
plt.figure(figsize=(50, 10))
sns.lineplot(x=data.index, y=data["Total"])
plt.xlabel("Day over time")
plt.ylabel("Total Marginal Value of Electricity")
plt.ylim(0, 1000)
plt.title("Total Value from 2020 to 2024")
plt.show()

### Feature Engineering

In [ ]:
data.head()

In [ ]:
# Adding Summer column
data["Summer"] = 0

# Set Summer = 1 where Month is July–October (7–10)
data.loc[data["Month"].isin([7, 8, 9, 10]), "Summer"] = 1

# Adding peak hours column if hour between 15 and 20
data["Peak"] = 0
data.loc[(data["Hour"] >= 15) & (data["Hour"] <= 20), "Peak"] = 1

In [ ]:
data.head()

---

### Trying Multivariable Regression

Predicting: `Total` energy value

Candidate independent variables:
- Year
- Month - Summer/not summer
- Hour
- Interaction variable between summer and peak hours
- Energy
- Generation Capacity
- Transmission
- Distribution
- Avoided AS Procurement
- GHG Cap and Trade
- GHG Adder


In [ ]:
# Changing Summer to categorical variable and Year to ordinal variable
data["Year"] = data["Year"].astype("category")
data["SummerxPeak"] = data["Summer"] * data["Peak"]

# Response variable
response = 'Total'

# Define covariates
covariates = ["Year", "Hour", "SummerxPeak", "Energy", "Generation", "Transmission", "Distribution"]


model_formula = f'{response} ~ {" + ".join(covariates)}'
print(f"Model Formula: {model_formula}")

# Splitting Data into training and testing sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Training Model on Training Data
full_model = smf.ols(model_formula, data=train_data).fit()

# Making Predictions on Test Set
y_test = test_data[response]  # Actual values
y_pred = full_model.predict(test_data)  # Predicted values

# Computing RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(full_model.summary())
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")


#### Checking for multicollinearity

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = data[["Year", "Summer", "Hour", "SummerxPeak", "Energy", "Generation", "Transmission", "Distribution"]]
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_data

No major issues with multicollinearity

---

### Testing similar models

In [ ]:
# Seeing which components make up the Total the most: Energy, Generation, Transmission, Distribution, Adder

# Response variable
response = 'Total'

data = data.rename(columns={
    "Avoided AS Procurement": "AncillaryServices",
    "GHG Cap and Trade": "CapTrade",
    "GHG Adder": "Adder"
})
# Define covariates
covariates = ["Energy", "Generation", "Transmission", "Distribution"]


model_formula = f'{response} ~ {" + ".join(covariates)}'
print(f"Model Formula: {model_formula}")

# Splitting Data into training and testing sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Training Model on Training Data
full_model = smf.ols(model_formula, data=train_data).fit()

# Making Predictions on Test Set
y_test = test_data[response]  # Actual values
y_pred = full_model.predict(test_data)  # Predicted values

# Computing RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(full_model.summary())
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")


Notes:
- Removing summer, year, and ancillary services entirely from model barely changes RMSE!
- CapTrade and SummerxPeak also doesn't affect RMSE too much when removed
- Distribution is not as important as generation or transmission!

In [ ]:
# Simple model with info that consumers would have access to

# Response variable
response = 'Total'

data = data.rename(columns={
    "Avoided AS Procurement": "AncillaryServices",
    "GHG Cap and Trade": "CapTrade",
    "GHG Adder": "Adder"
})
# Define covariates
covariates = ["Year", "Hour", "SummerxPeak", "Energy", "Generation"]


model_formula = f'{response} ~ {" + ".join(covariates)}'
print(f"Model Formula: {model_formula}")

# Splitting Data into training and testing sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Training Model on Training Data
full_model = smf.ols(model_formula, data=train_data).fit()

# Making Predictions on Test Set
y_test = test_data[response]  # Actual values
y_pred = full_model.predict(test_data)  # Predicted values

# Computing RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(full_model.summary())
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")


In [ ]:
X = data[["Year", "Hour", "SummerxPeak", "Energy", "Generation"]]
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_data

---

### Fixed Costs Model

In [ ]:
data = pd.read_excel("E3 ACC Data/CZ7 Fixed Data.xlsx")
data.head()

In [ ]:
columns = [col for col in data.columns if col not in ['Year', 'Month','Day','Hour']]

data['Total'] = data[columns].sum(axis=1)

plt.figure(figsize=(50, 20))
sns.lineplot(x=data.index, y=data["Total"])
plt.xlabel("Day over time")
plt.ylabel("Total Marginal Value of Electricity")
plt.title("Total Value from 2020 to 2024")
plt.show()

### Feature Engineering

In [ ]:
# Adding Summer column
data["Summer"] = 0

# Set Summer = 1 where Month is July–October (7–10)
data.loc[data["Month"].isin([7, 8, 9, 10]), "Summer"] = 1

# Adding peak hours column if hour between 15 and 20
data["Peak"] = 0
data.loc[(data["Hour"] >= 15) & (data["Hour"] <= 20), "Peak"] = 1

In [ ]:
data.head()

In [ ]:
# Full Model
data["Year"] = data["Year"].astype("category")
data["SummerxPeak"] = data["Summer"] * data["Peak"]


# Response variable
response = 'Total'

data = data.rename(columns={"GHG Portfolio Rebalancing": "GHGRebalancing",
                            "Methane Leakage": "MethaneLeak",
                            "GHG Adder": "Adder"
                           })

# Define covariates
covariates = ["Year", "Hour", "SummerxPeak", "GHGRebalancing", "Losses", "MethaneLeak", "Adder"]


model_formula = f'{response} ~ {" + ".join(covariates)}'
print(f"Model Formula: {model_formula}")

# Splitting Data into training and testing sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Training Model on Training Data
full_model = smf.ols(model_formula, data=train_data).fit()

# Making Predictions on Test Set
y_test = test_data[response]  # Actual values
y_pred = full_model.predict(test_data)  # Predicted values

# Computing RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(full_model.summary())
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")


In [ ]:
print(data[["GHGRebalancing", "Losses", "MethaneLeak", "Adder"]].corr())


In [ ]:
# Removing Adder because of strong correlation with GHGRebalancing and MethaneLeak

# Response variable
response = 'Total'

data = data.rename(columns={"GHG Portfolio Rebalancing": "GHGRebalancing",
                            "Methane Leakage": "MethaneLeak",
                            "GHG Adder": "Adder"
                           })

# Define covariates
covariates = ["GHGRebalancing", "Losses", "MethaneLeak"]


model_formula = f'{response} ~ {" + ".join(covariates)}'
print(f"Model Formula: {model_formula}")

# Splitting Data into training and testing sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Training Model on Training Data
full_model = smf.ols(model_formula, data=train_data).fit()

# Making Predictions on Test Set
y_test = test_data[response]  # Actual values
y_pred = full_model.predict(test_data)  # Predicted values

# Computing RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(full_model.summary())
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")


- Adder the most significant contribution to RMSE reduction

---
### Total Electricity Value Model

In [ ]:
data = pd.read_excel("E3 ACC Data/CZ7 Yearly Data.xlsx")
data.head()

In [ ]:
columns = [col for col in data.columns if col not in ['Year', 'Month','Day','Hour']]

data['Total'] = data[columns].sum(axis=1)

plt.figure(figsize=(50, 20))
sns.lineplot(x=data.index, y=data["Total"])
plt.xlabel("Day over time")
plt.ylabel("Total Marginal Value of Electricity")
plt.title("Total Value from 2020 to 2024")
plt.ylim(-100, 1000)
plt.show()

In [ ]:
# Adding Summer column
data["Summer"] = 0

# Set Summer = 1 where Month is July–October (7–10)
data.loc[data["Month"].isin([7, 8, 9, 10]), "Summer"] = 1

# Adding peak hours column if hour between 15 and 20
data["Peak"] = 0
data.loc[(data["Hour"] >= 15) & (data["Hour"] <= 20), "Peak"] = 1

### Model Creation

In [ ]:
# Full Model
data["Year"] = data["Year"].astype("category")
data["SummerxPeak"] = data["Summer"] * data["Peak"]


# Response variable
response = 'Total'

data = data.rename(columns={"GHG Portfolio Rebalancing": "GHGRebalancing",
                            "Methane Leakage": "MethaneLeak",
                            "GHG Adder": "Adder",
                             "Avoided AS Procurement": "AncillaryServices",
                            "GHG Cap and Trade": "CapTrade"
                           })

# Define covariates
covariates = ["Year", "Hour", "SummerxPeak", "Energy", "Generation", "Transmission", "Distribution", "GHGRebalancing", "Losses", "MethaneLeak", "Adder"]

model_formula = f'{response} ~ {" + ".join(covariates)}'
print(f"Model Formula: {model_formula}")

# Splitting Data into training and testing sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Training Model on Training Data
full_model = smf.ols(model_formula, data=train_data).fit()

# Making Predictions on Test Set
y_test = test_data[response]  # Actual values
y_pred = full_model.predict(test_data)  # Predicted values

# Computing RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(full_model.summary())
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")


In [ ]:
# Full Model
data["Year"] = data["Year"].astype("category")
data["SummerxPeak"] = data["Summer"] * data["Peak"]


# Response variable
response = 'Total'

data = data.rename(columns={"GHG Portfolio Rebalancing": "GHGRebalancing",
                            "Methane Leakage": "MethaneLeak",
                            "GHG Adder": "Adder",
                             "Avoided AS Procurement": "AncillaryServices",
                            "GHG Cap and Trade": "CapTrade"
                           })

# Define covariates
covariates = ["Hour", "Energy", "Generation", "Transmission", "Distribution", "MethaneLeak"]

model_formula = f'{response} ~ {" + ".join(covariates)}'
print(f"Model Formula: {model_formula}")

# Splitting Data into training and testing sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Training Model on Training Data
full_model = smf.ols(model_formula, data=train_data).fit()

# Making Predictions on Test Set
y_test = test_data[response]  # Actual values
y_pred = full_model.predict(test_data)  # Predicted values

# Computing RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(full_model.summary())
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = data[["Hour", "Energy", "Generation", "Transmission", "Distribution", "MethaneLeak"]]
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_data

---

In [ ]:
years = ["2020", "2021", "2022", "2024"]
# palette = ["red", "green", "blue", "purple"] 
palette = ["#bae161", "#FFC300", "#C70039", "#581845"] 
palette_dict = {"2019": "#bae161", "2021": "#FFC300", "2022": "#C70039", "2024": "#581845"}

yearly_data = [may19_data, may21_data, may22_data, may24_data]  # DataFrames for each year

# --- Scatter Plot ---
plt.figure(figsize=(8, 5))
for i, data in enumerate(yearly_data):  # Iterate over datasets
    color = palette[i]
    label = years[i]

    for col in data.columns[1:]:  # Exclude the first column if necessary
        sns.scatterplot(x=data.index, y=data[col], color=color, label=label if col == data.columns[1] else None, alpha=0.25)

plt.xlabel("Hour of the Day")
plt.ylabel("TLV (Avg Hourly Energy Value) in $/MWh")
plt.title("May Yearly TLV Scatter Plot")
plt.legend(title="Year", loc="upper left")
plt.show()

# --- Line Plot ---
plt.figure(figsize=(8, 5))
for i, data in enumerate(yearly_data):  # Iterate over datasets
    color = palette[i]
    label = years[i]

    for col in data.columns[1:]:  # Exclude the first column if necessary
        sns.lineplot(x=data.index, y=data[col], color=color, label=label if col == data.columns[1] else None, alpha=0.4)

plt.xlabel("Hour of the Day")
plt.ylabel("TLV (Avg Hourly Energy Value) in $/MWh")
plt.title("May Yearly TLV Line Plot")
plt.legend(title="Year", loc="upper left")
plt.show()


In [ ]:
# Add a "Year" column to each DataFrame
may19_data_temporal_melted["Year"] = "2019"
may21_data_temporal_melted["Year"] = "2021"
may22_data_temporal_melted["Year"] = "2022"
may24_data_temporal_melted["Year"] = "2024"

# Combine both datasets into one
yearly_temporal_data = [may19_data_temporal_melted, may21_data_temporal_melted, may22_data_temporal_melted, may24_data_temporal_melted]
combined_data = pd.concat(yearly_temporal_data)

plt.figure(figsize=(10, 6))

# Create stacked horizontal box plots (assign hue for color separation)
sns.boxplot(x="Value", y="Year", data=combined_data, hue="Year", palette=palette_dict, legend=False)

# Set labels and title
plt.xlabel("Hourly Total Levelized Value Distribution")
plt.ylabel("Year")
plt.title("May TLV Distribution by Year (Quartile Plot)")


plt.show()

plt.figure(figsize=(10, 6))
# Create stacked horizontal box plots (assign hue for color separation)
sns.boxplot(x="Value", y="Year", data=combined_data, hue="Year", palette=palette_dict, legend=False)

# Set labels and title
plt.xlabel("Hourly Total Levelized Value Distribution")
plt.ylabel("Year")
plt.title("May TLV Distribution by Year (Quartile Plot) Zoomed-In")
plt.xlim(-50, 300)

plt.show()


In [ ]:
plt.figure(figsize=(10, 6))

# Create histogram with transparency
sns.histplot(data=combined_data, x="Value", hue="Year", bins=100, stat="density", 
             palette = palette_dict, 
             alpha=0.3)

# Overlay PDF line using KDE plot with explicit labels
for i, data in enumerate(yearly_temporal_data):  # Iterate over datasets
    year = years[i]
    color = palette[i]
    subset = combined_data[combined_data["Year"] == year]
    sns.kdeplot(x=subset["Value"], color=color, linewidth=2, label=year)

# Set labels and title
plt.xlabel("Hourly Total Levelized Value Distribution")
plt.ylabel("Density")
plt.title("May TLV Histogram with PDF Line Zoomed-In")
plt.xlim(-50, 300)
plt.legend(title="Year")
plt.show()


#### Year to Year Comparisons

2019 to 2021

In [ ]:
years = ["2019", "2021"]
palette = ["red", "blue"] 

yearly_data = [may19_data, may21_data]  # DataFrames for each year

plt.figure(figsize=(10, 6))
for i, data in enumerate(yearly_data):  # Iterate over datasets
    color = palette[i]
    label = years[i]

    for col in data.columns[1:]:  # Exclude the first column if necessary
        sns.lineplot(x=data.index, y=data[col], color=color, label=label if col == data.columns[1] else None, alpha=0.4)

plt.xlabel("Hour of the Day")
plt.ylabel("TLV (Avg Hourly Energy Value) in $/MWh")
plt.title("Yearly TLV Line Plot")
plt.legend(title="Year", loc="upper left")
plt.show()


2021 to 2022

In [ ]:
years = ["2021", "2022"]
palette = ["red", "blue"] 

yearly_data = [may21_data, may22_data]  # DataFrames for each year

plt.figure(figsize=(10, 6))
for i, data in enumerate(yearly_data):  # Iterate over datasets
    color = palette[i]
    label = years[i]

    for col in data.columns[1:]:  # Exclude the first column if necessary
        sns.lineplot(x=data.index, y=data[col], color=color, label=label if col == data.columns[1] else None, alpha=0.4)

plt.xlabel("Hour of the Day")
plt.ylabel("TLV (Avg Hourly Energy Value) in $/MWh")
plt.title("Yearly TLV Line Plot")
plt.legend(title="Year", loc="upper left")
plt.show()


2022 to 2024

In [ ]:
years = ["2022", "2024"]
palette = ["red", "blue"] 

yearly_data = [may22_data, may24_data]  # DataFrames for each year

plt.figure(figsize=(10, 6))
for i, data in enumerate(yearly_data):  # Iterate over datasets
    color = palette[i]
    label = years[i]

    for col in data.columns[1:]:  # Exclude the first column if necessary
        sns.lineplot(x=data.index, y=data[col], color=color, label=label if col == data.columns[1] else None, alpha=0.4)

plt.xlabel("Hour of the Day")
plt.ylabel("TLV (Avg Hourly Energy Value) in $/MWh")
plt.title("Yearly TLV Line Plot")
plt.legend(title="Year", loc="upper left")
plt.show()


In [ ]:
# Initialize DataFrame for basic stats
yearly_distr = pd.DataFrame()

# List of datasets per year
years = ["2019", "2021", "2022", "2024"]

for i, data in enumerate(yearly_temporal_data):  # Iterate over datasets
    year = years[i]
    data_values = data["Value"].dropna()

    # Compute mean (μ) and standard deviation (σ)
    mu = np.mean(data_values)
    sigma = np.std(data_values, ddof=1)  # Sample std dev

    # Compute IQR for outlier detection
    Q1 = np.percentile(data_values, 25)
    Q3 = np.percentile(data_values, 75)
    IQR = Q3 - Q1

    # Define outlier bounds
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Count outliers
    outliers = data_values[(data_values < lower_bound) | (data_values > upper_bound)]
    num_outliers = len(outliers)

    # Store values in DataFrame
    yearly_distr.loc[year, "mean"] = mu
    yearly_distr.loc[year, "standard deviation"] = sigma
    yearly_distr.loc[year, "outlier count"] = num_outliers

# Calculating Marginal Difference Percentage
yearly_distr["% change in mean"] = yearly_distr["mean"].pct_change() * 100

print("Descriptive Stats: ")
yearly_distr

#### Creating Line of Best Fit Drawing for ALl Years

2019

In [ ]:
# Parameters
mu = np.linspace(0, 23, 7) # Uses 7 mu's equally spaced out
sigma = (max(x) - min(x)) / 10 #using (max(x) - min(x))/(# of basis functions) for sigma


w = gaussian_weights(x_19, y_19, mu, sigma) #Optimal weight vectors given x and y in dataset

print("sigma: ", sigma)
print("weight vector: ", w)

# Plotting H(x) prediction line on data
x_reg = np.linspace(0, 24, 100)
H_19= plot_h(x_reg, mu, w, sigma)

plt.plot(x_reg, H_19, color="red")
plt.scatter(x_19, y_19)

# Add labels and legend
plt.xlabel('x')
plt.ylabel('y')
plt.title('TLV by Hour Using Gaussian Basis Functions with Optimized Weights')
plt.show()


2021

In [ ]:
# Parameters
mu = np.linspace(0, 23, 7) # Uses 7 mu's equally spaced out
sigma = (max(x) - min(x)) / 10 #using (max(x) - min(x))/(# of basis functions) for sigma


w = gaussian_weights(x_21, y_21, mu, sigma) #Optimal weight vectors given x and y in dataset

print("sigma: ", sigma)
print("weight vector: ", w)

# Plotting H(x) prediction line on data
x_reg = np.linspace(0, 24, 100)
H_21 = plot_h(x_reg, mu, w, sigma)

plt.plot(x_reg, H_21, color="red")
plt.scatter(x_21, y_21)

# Add labels and legend
plt.xlabel('x')
plt.ylabel('y')
plt.title('TLV by Hour Using Gaussian Basis Functions with Optimized Weights')
plt.show()


2022

In [ ]:
# Parameters
mu = np.linspace(0, 23, 7) # Uses 7 mu's equally spaced out
sigma = (max(x) - min(x)) / 10 #using (max(x) - min(x))/(# of basis functions) for sigma


w = gaussian_weights(x_22, y_22, mu, sigma) #Optimal weight vectors given x and y in dataset

print("sigma: ", sigma)
print("weight vector: ", w)

# Plotting H(x) prediction line on data
x_reg = np.linspace(0, 24, 100)
H_22 = plot_h(x_reg, mu, w, sigma)

plt.plot(x_reg, H_22, color="red")
plt.scatter(x_22, y_22)

# Add labels and legend
plt.xlabel('x')
plt.ylabel('y')
plt.title('TLV by Hour Using Gaussian Basis Functions with Optimized Weights')
plt.show()


2024

In [ ]:
# Parameters
mu = np.linspace(0, 23, 7) # Uses 7 mu's equally spaced out
sigma = (max(x) - min(x)) / 10 #using (max(x) - min(x))/(# of basis functions) for sigma


w = gaussian_weights(x_24, y_24, mu, sigma) #Optimal weight vectors given x and y in dataset

print("sigma: ", sigma)
print("weight vector: ", w)

# Plotting H(x) prediction line on data
x_reg = np.linspace(0, 24, 100)
H_24= plot_h(x_reg, mu, w, sigma)

plt.plot(x_reg, H_24, color="red")
plt.scatter(x_24, y_24)

# Add labels and legend
plt.xlabel('x')
plt.ylabel('y')
plt.title('TLV by Hour Using Gaussian Basis Functions with Optimized Weights')
plt.show()


Plotting all Gaussian line of best fits

In [ ]:
line_plots = [H_19, H_21, H_22, H_24]

# --- Line Plot ---
plt.figure(figsize=(8, 5))
for i, data in enumerate(line_plots):  # Iterate over datasets
    color = palette[i]
    label = years[i]

    plt.plot(x_reg, line_plots[i], color=color, label=label)


plt.xlabel("Hour of the Day")
plt.ylabel("TLV (Avg Hourly Energy Value) in $/MWh")
plt.title("May Yearly TLV Line of Best Fit Plot w/ Gaussians")
plt.legend(title="Year", loc="upper left")
plt.show()


Regression Analysis

In [ ]:
# Define response variable
response = 'Value'

# Define covariates
covariates = ["Day_of_Month", "Year", "hour_sin", "hour_cos"]


model_formula = f'{response} ~ {" + ".join(covariates)}'
print(f"Model Formula: {model_formula}")

# Splitting Data into training and testing sets
train_data, test_data = train_test_split(may_yearly_data, test_size=0.2, random_state=42)

# Training Model on Training Data
full_model = smf.ols(model_formula, data=train_data).fit()

# Making Predictions on Test Set
y_test = test_data[response]  # Actual values
y_pred = full_model.predict(test_data)  # Predicted values

# Computing RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(full_model.summary())
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
